# Module 14: Distributed URL Shortener TinyURL — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/url_shortener_service.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import url_shortener_service

classes = [n for n, o in inspect.getmembers(url_shortener_service, inspect.isclass)
           if o.__module__ == 'url_shortener_service']
functions = [n for n, o in inspect.getmembers(url_shortener_service, inspect.isfunction)
             if o.__module__ == 'url_shortener_service']

print('module   : url_shortener_service')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(url_shortener_service, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Base62 bijective symmetry

This is the module's own `test_base62_bijective_symmetry` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
import time

import pytest
from url_shortener_service import (
    Base62Encoder,
    KeyGenerationService,
    URLShortenerService,
)

test_numbers = [0, 1, 61, 62, 125, 999_999, 100_000_000, 3_500_000_000_000]

for num in test_numbers:
    encoded = Base62Encoder.encode(num, min_length=7)
    assert len(encoded) >= 7
    decoded = Base62Encoder.decode(encoded)
    assert decoded == num

print('PASSED: test_base62_bijective_symmetry')

## 3. 🔮 Prediction — commit before you run

Predict how many characters a base62 short code needs to address 100 billion URLs. Then predict the collision probability if codes were random rather than sequential.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_url_shorten_and_resolve_lifecycle`, which tests exactly this property.


In [ ]:
kgs = KeyGenerationService(start_id=100_000)
service = URLShortenerService(kgs)

original = "https://www.google.com/search?q=system+design+mastery"
short_url = service.shorten_url(original)

assert short_url.startswith("https://tiny.url/")
token = short_url.split("/")[-1]
assert len(token) == 7

# Resolve URL
resolved = service.resolve_url(short_url)
assert resolved == original

# Analytics check
stats = service.get_analytics(token)
assert stats["clicks"] == 1

# Second click
service.resolve_url(short_url)
stats2 = service.get_analytics(token)
assert stats2["clicks"] == 2

print('PASSED: test_url_shorten_and_resolve_lifecycle')

## 4. Measure it: Ttl expiration behavior

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_ttl_expiration_behavior` and times it.


In [ ]:

_t0 = time.perf_counter()

kgs = KeyGenerationService()
service = URLShortenerService(kgs)

# 0.2 second TTL
short_url = service.shorten_url("https://fast-expire.com", ttl_seconds=0.2)
assert service.resolve_url(short_url) == "https://fast-expire.com"

time.sleep(0.3)  # wait for expiration

with pytest.raises(KeyError, match="does not exist or expired"):
    service.resolve_url(short_url)

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_ttl_expiration_behavior')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(url_shortener_service) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. Key length is a capacity calculation: log_62(address space).
2. Sequential-plus-encode avoids collision handling that random codes require.
3. The read:write ratio decides your entire caching strategy.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
